# CreditLens — 02 · Feature Engineering

**You write the code cells; I review.** Goal: turn raw tables into one **model-ready frame** —
one row per `SK_ID_CURR`, numeric/categorical features, no target leakage.

Once the logic here is proven, we refactor it into `creditlens/data/features.py` (callable by the pipeline).

Plan:
1. Application-level features (ratios + age/employment).
2. Aggregate `bureau` → one row per applicant.
3. Aggregate `previous_application` → one row per applicant.
4. Join everything onto the application frame (left join — keep every applicant).
5. Sanity-check: row count unchanged, target still present and untouched.

## 0 · Setup

In [ ]:
# TODO: sys.path.insert(0, '..'); import pandas/numpy
#       from creditlens.data.load import load_application, load_bureau, load_previous_application
#       from creditlens.config import TARGET, ID_COL
#       app = load_application()


## 1 · Application features
Add affordability ratios + decoded time features. Keep `SK_ID_CURR` and `TARGET` intact.
Suggested: `CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `CREDIT_TERM = AMT_ANNUITY/AMT_CREDIT`,
`AGE_YEARS = -DAYS_BIRTH/365`, `EMPLOYED_YEARS = -DAYS_EMPLOYED/365` (already NaN-safe from loader),
`EMPLOYED_TO_AGE = DAYS_EMPLOYED / DAYS_BIRTH`.
**Watch:** division by zero / NaN — decide how to handle (np.inf → NaN).

In [ ]:
# TODO: def add_application_features(df): ... return df


## 2 · Aggregate `bureau`
Many rows per applicant → collapse to one. Useful aggregations:
- count of past bureau credits
- count where `CREDIT_ACTIVE == 'Active'`
- sum/mean of `AMT_CREDIT_SUM`, `AMT_CREDIT_SUM_DEBT`
- sum of `CREDIT_DAY_OVERDUE`, max `CREDIT_DAY_OVERDUE`

Name columns with a `BUREAU_` prefix so they're traceable after the join.

In [ ]:
# TODO: def aggregate_bureau(bureau): groupby(ID_COL).agg({...}); flatten columns; add BUREAU_ prefix


## 3 · Aggregate `previous_application`
Collapse to one row per applicant. Useful:
- count of previous applications
- approval rate (`NAME_CONTRACT_STATUS == 'Approved'`)
- mean/max of `AMT_APPLICATION`, `AMT_CREDIT`
- mean `CNT_PAYMENT`

Prefix `PREV_`.

In [ ]:
# TODO: def aggregate_previous(prev): groupby(ID_COL).agg({...}); flatten; PREV_ prefix


## 4 · Join
Left-join the two aggregates onto the application frame on `SK_ID_CURR`.
Applicants with no bureau/previous history get NaN — that's expected (impute later).

In [ ]:
# TODO: df = app.merge(bureau_agg, on=ID_COL, how='left').merge(prev_agg, on=ID_COL, how='left')


## 5 · Sanity checks (catch leakage / mistakes)
- row count == original application row count (left join didn't duplicate)
- `TARGET` present, still binary, unchanged distribution
- `SK_ID_CURR` still unique
- no feature equals/derives from `TARGET`
- report new column count + missingness of the new BUREAU_/PREV_ columns

In [ ]:
# TODO: asserts on shape[0], TARGET, SK_ID_CURR uniqueness; print final shape


## Next
Once this runs clean, we move it into `creditlens/data/features.py` as
`build_features(app, bureau, prev) -> DataFrame`, then Phase 3 (models) consumes it.